In [1]:
import torch
import os
import numpy as np
from torch import nn
from monai.data import DataLoader, Dataset, CacheDataset, decollate_batch
from monai.losses import DiceCELoss, HausdorffDTLoss, TverskyLoss
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from monai.networks.nets import UNet
import lightning as L
from torch.optim import Adam
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau
from glob import glob
import matplotlib.pyplot as plt
from monai.metrics import DiceMetric
from monai.visualize import plot_2d_or_3d_image
from monai.visualize.utils import blend_images
import monai.transforms as t

In [2]:

class LitUNet(L.LightningModule):
    def __init__(self,
                 n_classes,
                 num_input_channel,
                 dropout=0.5,
                 check_val=10,
                 max_epochs=50,
                 optim_lr=0.001,
                 optim_betas=(0.9, 0.999),
                 optim_eps=1e-08,
                 optim_weight_decay=0,
                 optim_amsgrad=True):
        super().__init__()
        self.save_hyperparameters()

        self.model = UNet(
            spatial_dims=2,
            in_channels=num_input_channel,
            out_channels=n_classes,
            channels=(64, 128, 256, 512),
            strides=(2, 2, 2),
            dropout=dropout,
            num_res_units=2
        )

        self.loss = TverskyLoss(to_onehot_y=True, softmax=True)
        self.dice_metric = DiceMetric(
            include_background=False, reduction="mean", get_not_nans=False
        )

        self.post_pred = t.Compose(
            [t.EnsureType(), t.AsDiscrete(argmax=True, to_onehot=n_classes)])
        self.post_label = t.AsDiscrete(to_onehot=n_classes)

        self.train_step_outputs = []
        self.val_step_outputs = []
        self.train_losses = []
        self.val_losses = []
        self.metric_values = []
        self.best_val_dice = 0
        self.best_val_epoch = 0

    def forward(self, inputs):
        return self.model(inputs)

    def training_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self.model(images)
        loss = self.loss(outputs, labels)
        pred = {"train_loss": loss, "train_number": len(labels)}
        self.train_step_outputs.append(pred)
        return {"loss": loss}

    def on_train_epoch_end(self):
        outputs = self.train_step_outputs
        train_loss, num_items = 0, 0
        for output in outputs:
            train_loss += output["train_loss"].sum().item()
            num_items += output["train_number"]

        mean_train_loss = torch.tensor(train_loss / num_items)
        self.log("train_loss", mean_train_loss)
        self.train_losses.append(mean_train_loss.item())
        self.train_step_outputs.clear()  # free memory

    def validation_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self.model(images)
        loss = self.loss(outputs, labels)
        outputs = [self.post_pred(i) for i in decollate_batch(outputs)]
        labels = [self.post_label(i) for i in decollate_batch(labels)]

        self.dice_metric(y_pred=outputs, y=labels)

        pred = {"val_loss": loss,
                "val_number": len(outputs)}  # Save all losses
        self.val_step_outputs.append(pred)

        if batch_idx == 25:
            # blend_images(images[0], labels[0])
            plot_2d_or_3d_image(images*255, self.current_epoch + 1,
                                self.logger.experiment, tag="image")
            plot_2d_or_3d_image(labels, self.current_epoch + 1,
                                self.logger.experiment, tag="label")
            plot_2d_or_3d_image(outputs, self.current_epoch + 1,
                                self.logger.experiment, tag="output")
        return {"val_loss": loss, "val_number": len(outputs)}

    def on_validation_epoch_end(self):
        outputs = self.val_step_outputs
        val_loss, num_items = 0, 0

        for output in outputs:
            val_loss += output["val_loss"].sum().item()
            num_items += output["val_number"]
        mean_val_loss = torch.tensor(val_loss / num_items)
        self.val_losses.append(mean_val_loss.item())

        mean_val_dice = self.dice_metric.aggregate().item()

        self.dice_metric.reset()
        self.val_step_outputs.clear()  # free memory

        if mean_val_dice > self.best_val_dice:
            self.best_val_dice = mean_val_dice
            self.best_val_epoch = self.current_epoch

        print(
            f"current epoch: {self.current_epoch} "
            f"current mean dice: {mean_val_dice:.4f}"
            f"\nbest mean dice: {self.best_val_dice:.4f} "
            f"at epoch: {self.best_val_epoch}"
        )

        self.metric_values.append(mean_val_dice)
        self.log("val_dice", mean_val_dice)
        self.log("val_loss", mean_val_loss)
        self.val_step_outputs.clear()  # free memory

        tensorboard_logs = {
            "val_dice": mean_val_dice,
            "val_loss": mean_val_loss
        }
        return {"log": tensorboard_logs}

    def configure_optimizers(self):

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.hparams.optim_lr,
            betas=self.hparams.optim_betas,
            eps=self.hparams.optim_eps,
            weight_decay=self.hparams.optim_weight_decay,
            amsgrad=self.hparams.optim_amsgrad
        )
        return {
            "optimizer": self.optimizer,
            "lr_scheduler": {
                "scheduler": ReduceLROnPlateau(self.optimizer, factor=0.1, patience=5),
                "frequency": self.hparams.check_val,
                "monitor": "val_loss",
                "interval": "epoch",
            }
        }

    def plot_losses(self):
        plt.figure(figsize=(10, 5))
        plt.plot([i for i in range(len(self.train_losses))],
                 self.train_losses, label='Train Loss')
        plt.plot([i * self.hparams.check_val for i in range(len(self.val_losses))],
                 self.val_losses, label='Validation Loss')
        plt.xlabel('Iterations')
        plt.ylabel('Loss')
        plt.title('Training and Validation Losses')
        plt.legend()
        plt.show()

In [3]:
best_model_path = r'tb_logs\cell_unet_model_small\version_0\checkpoints\checkpoint-cell_seg_unet-epoch=239-val_loss=0.00.ckpt'
best_model = LitUNet.load_from_checkpoint(best_model_path)

In [4]:
unwrapped_best_model = best_model.model

In [5]:
import monai.networks.utils as m_nn_utils

In [6]:
import onnx
import onnxruntime

In [7]:
with torch.no_grad():
    with m_nn_utils.eval_mode(unwrapped_best_model):
        unwrapped_best_model.eval()
        unwrapped_best_model.cpu()

        dummy_input_1 = torch.randn(
            (1, 1, 192, 192), requires_grad=False, device='cpu')

        torch_out_1 = unwrapped_best_model(dummy_input_1)

        ts_model = torch.export.export(
            unwrapped_best_model,
            args=(dummy_input_1,),
            strict=True
        )

        ts_out_1 = ts_model.module()(dummy_input_1)
        np.testing.assert_allclose(torch_out_1.detach().cpu().numpy(
        ), ts_out_1.detach().cpu().numpy(), rtol=0.01, atol=0.001)

        onnx_program = torch.onnx.export(
            ts_model,
            (dummy_input_1,),
            f="model.onnx",
            opset_version=22,
            input_names=["input"],
            output_names=["output"],
            optimize=True,
            external_data=False,
            # verify=True,
            dynamo=True,
        )

        onnx_model = onnx.load("model.onnx")
        onnx.checker.check_model(onnx_model)
        ort_session = onnxruntime.InferenceSession("model.onnx")

        ort_inputs_1 = {ort_session.get_inputs(
        )[0].name: dummy_input_1.detach().cpu().numpy()}
        ort_outs_1 = ort_session.run(None, ort_inputs_1)
        np.testing.assert_allclose(
            torch_out_1.detach().cpu().numpy(), ort_outs_1[0], rtol=0.001, atol=0.001)

[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


In [8]:
np.testing.assert_allclose(
            torch_out_1.detach().cpu().numpy(), ort_outs_1[0], rtol=0.001, atol=0.001)

In [9]:
onnx_model = onnx.load("./model.onnx")
onnx.checker.check_model(onnx_model)

In [10]:
ort_session = onnxruntime.InferenceSession("./model.onnx")